In [4]:
import os, re, glob
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.warp import transform_bounds
from shapely.geometry import box
from tqdm import tqdm

# =========================
# CONFIG (YOUR PATHS)
# =========================
FLOODS_BASE = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps"

COMMUNES_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\adminexpress-cog-simpl-000-2025.gpkg"
COMMUNES_LAYER = "commune"

COMMUNE_NAME_COL = "nom"       # adjust if uppercase in your file
COMMUNE_CODE_COL = "insee_com" # adjust if uppercase in your file

In [5]:
# =========================
# LOAD COMMUNES (AdminExpress)
# =========================
communes = gpd.read_file(COMMUNES_GPKG, layer=COMMUNES_LAYER)
communes = communes[communes.geometry.notna() & ~communes.geometry.is_empty].copy()

In [11]:
communes["insee_com"]

0        70177
1        89050
2        60414
3        80756
4        14478
         ...  
34872    97224
34873    97206
34874    97234
34875    97701
34876    97801
Name: insee_com, Length: 34877, dtype: object

In [20]:
!pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy

In [14]:
num = 55064  # the number you want to check

if (communes["insee_com"] == num).any():
    print("Yes, there is at least one row with this value.")
else:
    print("No, this value is not present.")

No, this value is not present.


In [21]:
import geopandas as gpd
import pandas as pd
import fiona
import os

# Input GeoPackage
COMMUNES_GPKG = r"D:\M2_MoSEF\Acclim\DataCollection\data\adminexpress-cog-simpl-000-2025.gpkg"

# Output Excel file
output_excel = r"D:\M2_MoSEF\Acclim\DataCollection\data\adminexpress-cog-simpl-000-2025_layers.xlsx"

# List all layers in the GeoPackage
layers = fiona.listlayers(COMMUNES_GPKG)
print(f"Found {len(layers)} layers:", layers)

with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
    for layer in layers:
        print(f"Exporting layer: {layer}")
        gdf = gpd.read_file(COMMUNES_GPKG, layer=layer)

        # Convert geometry to WKT so Excel can store it
        if "geometry" in gdf.columns:
            gdf["geometry"] = gdf.geometry.to_wkt()

        # Excel sheet names max length = 31
        sheet_name = layer[:31]

        # Write to Excel
        gdf.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"✅ Export finished: {output_excel}")

Found 9 layers: ['commune_int', 'commune', 'departement', 'region', 'cheflieu', 'epci', 'departement_int', 'region_int', 'epci_int']
Exporting layer: commune_int


C:\Users\Juan David Alonso\AppData\Local\Temp\ipykernel_2596\3863467336.py:23: UserWarning: Geometry column does not contain geometry.
  gdf["geometry"] = gdf.geometry.to_wkt()


Exporting layer: commune


C:\Users\Juan David Alonso\AppData\Local\Temp\ipykernel_2596\3863467336.py:23: UserWarning: Geometry column does not contain geometry.
  gdf["geometry"] = gdf.geometry.to_wkt()


Exporting layer: departement
Exporting layer: region
Exporting layer: cheflieu


...

Exporting layer: epci


C:\Users\Juan David Alonso\AppData\Local\Temp\ipykernel_2596\3863467336.py:23: UserWarning: Geometry column does not contain geometry.
  gdf["geometry"] = gdf.geometry.to_wkt()
C:\Users\Juan David Alonso\AppData\Local\Temp\ipykernel_2596\3863467336.py:23: UserWarning: Geometry column does not contain geometry.
  gdf["geometry"] = gdf.geometry.to_wkt()
C:\Users\Juan David Alonso\AppData\Local\Temp\ipykernel_2596\3863467336.py:23: UserWarning: Geometry column does not contain geometry.
  gdf["geometry"] = gdf.geometry.to_wkt()


Exporting layer: departement_int
Exporting layer: region_int
Exporting layer: epci_int


C:\Users\Juan David Alonso\AppData\Local\Temp\ipykernel_2596\3863467336.py:23: UserWarning: Geometry column does not contain geometry.
  gdf["geometry"] = gdf.geometry.to_wkt()


✅ Export finished: D:\M2_MoSEF\Acclim\DataCollection\data\adminexpress-cog-simpl-000-2025_layers.xlsx
